In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/return_risk_chargeback_dataset.csv')

print(df.shape)
print(df['is_returned'].mean())
df.head()

(20000, 19)
0.21195


,customer_id,customer_past_returns,customer_past_chargebacks,product_category,item_price,quantity,order_value,payment_method,payment_attempts,order_date,delivery_days,delivery_date,billing_country,shipping_country,address_mismatch,is_returned,days_to_return,return_date,is_chargeback
0,4174,1,0,groceries,766.884816,3,2300.654448,upi,1,2025-10-11,10,2025-10-21,UAE,UAE,0,0,NaN,NaN,0
1,4507,0,0,electronics,13014.854628,1,13014.854628,credit_card,3,2025-02-26,4,2025-03-02,India,India,0,0,NaN,NaN,0
2,1860,1,0,home_furniture,2078.900125,3,6236.700375,upi,1,2025-08-13,5,2025-08-18,India,India,0,0,NaN,NaN,0
3,2294,1,0,apparel,987.297130,1,987.297130,cod,1,2025-07-28,2,2025-07-30,India,India,0,1,16.0,2025-08-15,0
4,2130,1,0,apparel,465.180864,1,465.180864,upi,3,2025-05-21,6,2025-05-27,UK,UK,0,0,NaN,NaN,1


In [4]:
from sklearn.model_selection import train_test_split

feature_cols = [
    'customer_past_returns', 'customer_past_chargebacks', 'product_category',
    'item_price', 'quantity', 'order_value', 'payment_method', 'payment_attempts',
    'delivery_days', 'address_mismatch'
]

X = df[feature_cols]
y = df['is_returned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train return rate:", y_train.mean())
print("Test return rate:", y_test.mean())

Train shape: (16000, 10) Test shape: (4000, 10)
Train return rate: 0.2119375
Test return rate: 0.212


In [5]:
X_train_encoded = pd.get_dummies(X_train, columns=['product_category', 'payment_method'], drop_first=False)
X_test_encoded = pd.get_dummies(X_test, columns=['product_category', 'payment_method'], drop_first=False)

X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print(X_train_encoded.columns.tolist())
print(X_train_encoded.shape, X_test_encoded.shape)

['customer_past_returns', 'customer_past_chargebacks', 'item_price', 'quantity', 'order_value', 'payment_attempts', 'delivery_days', 'address_mismatch', 'product_category_apparel', 'product_category_books', 'product_category_electronics', 'product_category_groceries', 'product_category_home_furniture', 'payment_method_cod', 'payment_method_credit_card', 'payment_method_debit_card', 'payment_method_netbanking', 'payment_method_upi']
(16000, 18) (4000, 18)


In [6]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train_encoded, y_train)

print("Model trained.")

Model trained.


In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test_encoded)
y_pred_proba = model.predict_proba(X_test_encoded)[:, 1]  # probability of class 1 (returned)

print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

              precision    recall  f1-score   support

           0       0.87      0.74      0.80      3152
           1       0.37      0.59      0.46       848

    accuracy                           0.70      4000
   macro avg       0.62      0.66      0.63      4000
weighted avg       0.76      0.70      0.72      4000

Confusion matrix:
 [[2320  832]
 [ 350  498]]
ROC-AUC: 0.7213892721602337


In [8]:
from sklearn.metrics import precision_recall_curve, f1_score

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# compute F1 at each threshold to find the best balance point
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)
print("At best threshold -> Precision:", precisions[best_idx], "Recall:", recalls[best_idx], "F1:", f1_scores[best_idx])

# apply this threshold and re-check confusion matrix
y_pred_tuned = (y_pred_proba >= best_threshold).astype(int)
print("\nConfusion matrix at tuned threshold:\n", confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))

Best threshold: 0.4250262858036355
At best threshold -> Precision: 0.33923630326508025 Recall: 0.722877358490566 F1: 0.4617702443863268

Confusion matrix at tuned threshold:
 [[1958 1194]
 [ 235  613]]
              precision    recall  f1-score   support

           0       0.89      0.62      0.73      3152
           1       0.34      0.72      0.46       848

    accuracy                           0.64      4000
   macro avg       0.62      0.67      0.60      4000
weighted avg       0.78      0.64      0.68      4000



In [9]:
importances = pd.Series(model.feature_importances_, index=X_train_encoded.columns).sort_values(ascending=False)
print(importances)

customer_past_returns              0.208105
item_price                         0.160503
product_category_apparel           0.159799
order_value                        0.152874
delivery_days                      0.066622
product_category_groceries         0.063579
quantity                           0.034718
product_category_books             0.030796
payment_attempts                   0.023831
customer_past_chargebacks          0.017793
payment_method_debit_card          0.012552
address_mismatch                   0.011674
payment_method_upi                 0.011514
payment_method_netbanking          0.011230
payment_method_credit_card         0.010987
payment_method_cod                 0.008500
product_category_electronics       0.007766
product_category_home_furniture    0.007158
dtype: float64


In [10]:
import joblib

joblib.dump(model, '../outputs/return_risk_model.pkl')
joblib.dump(list(X_train_encoded.columns), '../outputs/return_risk_model_columns.pkl')

print("Model and columns saved.")

Model and columns saved.
